# TaNP Gating-FiLM: три датасета на Colab T4
Ноутбук подготавливает данные и последовательно обучает отдельную модель для LastFM, MovieLens-1M и Gowalla. Все вычисления запускаются через `run_experiment.py`.

## Постоянное хранилище
Checkpoints, логи, таблицы и графики будут записываться прямо в Google Drive. Завершённые датасеты сохранятся даже при перезапуске Colab.

In [ ]:
from google.colab import drive
import json
drive.mount('/content/drive')
DRIVE_ROOT = Path('/content/drive/MyDrive/tanp_gating_film_seed2020')
DRIVE_ROOT.mkdir(parents=True, exist_ok=True)
with open(PROJECT / 'configs/tanp_gating_film_colab.json', encoding='utf-8') as stream:
    drive_config = json.load(stream)
drive_config['common']['output_dir'] = str(DRIVE_ROOT / 'artifacts/runs')
drive_config['common']['results_dir'] = str(DRIVE_ROOT / 'results')
drive_config['common']['figures_dir'] = str(DRIVE_ROOT / 'figures')
DRIVE_CONFIG = Path('/content/tanp_gating_film_colab_drive.json')
DRIVE_CONFIG.write_text(json.dumps(drive_config, ensure_ascii=False, indent=2), encoding='utf-8')
print('Результаты будут сохранены в:', DRIVE_ROOT)

In [ ]:
from pathlib import Path
REPO_URL = 'https://github.com/vladimirlavrentyev/cu-cold-start-recommendation-research.git'
PROJECT = Path('/content/cu-cold-start-recommendation-research')
if PROJECT.exists():
    !git -C {PROJECT} pull --ff-only
else:
    !git clone {REPO_URL} {PROJECT}
%cd {PROJECT}

In [ ]:
import torch
assert torch.cuda.is_available(), 'В Colab выберите Runtime → Change runtime type → T4 GPU'
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

In [ ]:
!python -m pip install -q -r requirements-colab.txt

## Подготовка данных
LastFM извлекается из авторского архива. MovieLens-1M загружается с GroupLens. Gowalla строится детерминированно из NGCF; точный подготовленный сплит Gowalla авторы TaNP не опубликовали, поэтому это реконструкция протокола.

In [ ]:
!python scripts/prepare_datasets.py --dataset all --seed 2020 --gowalla-max-users 2692

In [ ]:
!python run_experiment.py --suite {DRIVE_CONFIG} --dry-run

## Обучение
Следующая ячейка обучает три модели последовательно. При повторном запуске `--skip-existing` не переобучает уже завершённые датасеты.

In [ ]:
!python run_experiment.py --suite {DRIVE_CONFIG} --skip-existing

## Проверка сохранённых результатов

In [ ]:
print('Каталог результатов:', DRIVE_ROOT)
for path in sorted(DRIVE_ROOT.rglob('*')):
    if path.is_file():
        print(path.relative_to(DRIVE_ROOT))